# Fase 1 — Business Understanding

**Objetivo:** entender o problema de negócio, prever `SeriousDlqin2yrs` (se o business sofrerá inadimplência grave, atraso de 90+ dias, nos próximos 2 anos).  

Dataset: [Give Me Some Credit](https://www.kaggle.com/competitions/GiveMeSomeCredit/overview) (Kaggle).  

## 1. Contexto de Negócio  

Instituições financeiras: bancos tradicionais, fintechs de crédito e credit unions, concedem empréstimos, cartões de crédito e linhas de crédito rotativo para milhões de clientes todos os anos. Cada decisão de concessão carrega um dilema estrutural: conceder crédito demais para clientes de alto risco gera inadimplência (perda direta de capital); conceder crédito de menos para clientes de baixo risco gera perda de receita e clientes migrando para concorrentes.  

Esse dilema ficou brutalmente evidente durante a crise financeira de 2008, quando modelos de risco subestimaram sistematicamente a probabilidade de inadimplência em massa nos EUA, um dos gatilhos da crise de subprime. É justamente nesse contexto pós-crise que o dataset "Give Me Some Credit" foi disponibilizado (competição Kaggle de 2011, patrocinada pela Credit Fusion), com o objetivo de estimular o desenvolvimento de modelos melhores de probabilidade de inadimplência (Probability of Default — PD).  

Hoje, a área de Risk Management / Credit Risk de qualquer instituição financeira depende de scores preditivos para decidir, em segundos, se aprova ou não um crédito, e em qual condição. Atualmente, muitas decisões ainda se apoiam fortemente em:  

Scores de bureau tradicionais (como o FICO Score nos USA), que são "caixas-pretas" externas, caras e nem sempre bem calibradas para o público específico da instituição;  
Regras manuais e comitês de crédito, que são lentos e inconsistentes;  
Modelos legados, treinados há anos, sem recalibração para o comportamento atual dos consumidores.  

Isso limita a capacidade da empresa de:  
- Personalizar o risco por cliente, em vez de aplicar cortes genéricos;
- Precificar corretamente o crédito (risk-based pricing), cobrando juros proporcionais ao risco real;  
- Provisionar perdas com precisão, atendendo exigências contábeis e regulatórias (CECL nos EUA, IFRS 9 internacionalmente);  
- Escalar a concessão de crédito sem escalar proporcionalmente o time de analistas humanos.  

O papel da Ciência de Dados aqui é transformar o histórico financeiro do cliente (utilização de crédito, atrasos passados, renda, dívidas, número de linhas de crédito) em uma probabilidade objetiva e auditável de inadimplência grave nos próximos 2 anos, permitindo decisões de crédito mais rápidas, justas e rentáveis.

## 2. O problema de negócio

> Prever a probabilidade de um tomador de empréstimo sofrer inadimplência grave (90+ dias de atraso) nos próximos 2 anos, a partir do seu histórico financeiro atual  e antes de conceder ou renovar uma linha de crédito.

A variável-alvo do dataset é SeriousDlqin2yrs (1 = o cliente ficou 90+ dias em atraso em algum momento nos 2 anos seguintes; 0 = não ficou). Isso é literalmente o mesmo tipo de decisão que sustenta o negócio de empresas como Capital One, Discover, Upstart, Affirm e Klarna no mercado americano e o Nubank no mercado brasileiro, cujo motor de crédito (o "NuScore") é a peça central do modelo de negócio: eles conseguiram crescer oferecendo crédito a públicos "invisíveis" para os bureaus tradicionais justamente porque construíram um scoring proprietário mais granular que o mercado.

## 3. Por que Credit Risk Scoring é estratégico?

**Impacto Financeiro Direto (Perdas por Inadimplência)**

Cada ponto percentual de melhora na capacidade preditiva do modelo se traduz diretamente em menos capital perdido em write-offs. Diferente de um problema de marketing (onde o erro custa uma oportunidade), aqui o erro de Falso Negativo (aprovar quem vai inadimplir) é uma perda financeira direta e mensurável, e o erro de Falso Positivo (negar quem pagaria) é uma perda de receita por juros não cobrados. Esse trade-off é o coração do problema, é por isso que, em risk scoring, a escolha da métrica de avaliação (não apenas Accuracy, mas AUC-ROC, KS Statistic e o custo assimétrico dos erros) é uma decisão de negócio, não só técnica.

**Impacto Regulatório e de Compliance**

No mercado americano, modelos de crédito são regulados por leis como o Equal Credit Opportunity Act (ECOA) e o Fair Credit Reporting Act (FCRA), além de exigências contábeis como o CECL (Current Expected Credit Losses), que obriga bancos a provisionar perdas esperadas de forma antecipada e baseada em modelos. Isso significa que um modelo de crédito precisa ser auditável, explicável e não-discriminatório (não pode usar, direta ou indiretamente, variáveis como raça, gênero ou CEP como proxy discriminatório). Esse é um dos motivos pelos quais, mesmo em 2026, Regressão Logística e modelos baseados em árvore com SHAP ainda são tão usados em produção nesse setor — mais do que redes neurais complexas: a explicabilidade é requisito de negócio, não luxo técnico.

**Impacto Competitivo (Velocidade e Personalização)**

Instituições que conseguem decidir crédito em segundos, com scores mais granulares, conseguem capturar clientes bons mais rápido e evitar clientes ruins com mais precisão do que concorrentes que dependem só do bureau externo. Isso é exatamente a tese de negócio de fintechs como Upstart (usa +1600 variáveis, incluindo dados alternativos) e Nubank: quem tem o melhor modelo de risco consegue crescer a carteira de crédito com menor taxa de perda, ou seja, cresce de forma mais lucrativa que o mercado.

## 4. Segmentação de risco

Os clientes são segmentados por faixa de nota, no crédito os tomadores são segmentados por faixa de probabilidade de inadimplência (PD), o que direciona diretamente a decisão de negócio:  

| Categoria de Risco | Faixa de PD (exemplo) | Decisão de negócio típica |
|---|---|---|
| **Prime (baixo risco)** | PD baixa (ex.: < 5%) | Aprovação automática, melhores taxas de juros, maior limite |
| **Near-Prime (risco médio)** | PD intermediária (ex.: 5% < PD < 30%) | Aprovação condicional, limite reduzido, taxa ajustada ao risco |
| **Subprime (alto risco)** | PD alta (ex.: > 30%)| Negação, ou aprovação com garantias/juros altos, ou envio para análise manual |  


Cada faixa mal calibrada tem custo financeiro direto e quantificável 
em dólares, o que muda como você vai avaliar o modelo na Fase 5 
(Evaluation) do CRISP-DM.

**Como isso se relaciona com nosso projeto:**
- Nosso modelo vai gerar probabilidades (PD) contínuas de 0 a 1
- Essas probabilidades serão usadas para segmentar clientes nessas faixas
- O threshold de decisão (definido nas próximas seções) determinará 
  onde traçamos a linha entre "aprovar" e "negar"

## 5. Perguntas-guia

Perguntas guias que vão nos orientar e ajudar a modelar o problema. 

- Quem usaria esse modelo e para quê? (ex: decisão de aprovação de crédito)  
> Qualquer instituição financeira ou modelo de negócio que empresta dinheiro e queira prever se deve ou não emprestar dinheiro para algum cliente com o risco de sofrer default (modelo de aprovação de crédito).    
- O que significa, na prática, "prever probabilidade de inadimplência séria em 2 anos"?
> Prever se um cliente tomador de crédito terá inadimplência grave com um atraso de 90 dias ou mais nos próximos dois anos e qual o risco a empresa está tomando ao emprestar dinheiro para esse cliente. 

- Quais seriam os custos de um Falso Positivo (negar crédito a um bom pagador) vs. um Falso Negativo (aprovar crédito a um mau pagador)? Isso deveria influenciar como você otimiza o modelo depois?
>  Os custos de aprovar um Falso Negativo seriam muito maiores (aprovar crédito a um mau pagador), afinal teria probabilidade maior de calote. Negar um Falso Positivo não nos traria lucro e um Falso Negativo nos traria prejuízo. Isso influencia quais métricas vamos avaliar no modelo.  

- Que critério de sucesso você vai definir para o seu projeto de aprendizado (ex: AUC mínimo, Precision, Recall ou F1-Score)?
> Recall máximo é a métrica que deve ser priorizada nesse case, devemos minimizar os Falsos Negativos ao máximo (aprovar crédito para mau pagador) ou seja, para minimizar FN, devemos maximizar o Recall.  Recall mede sua capacidade de "não deixar nada escapar".  

>Recall = TP / (TP + FN)    

>Se FN ↑ (aumenta) → Recall ↓ (diminui)  
>Se FN ↓ (diminui) → Recall ↑ (aumenta)  


- O que o modelo precisa prever?   
> Se o cliente vai dar calote nos próximos dois anos: Sim(1) ou Não(0) na qual está na variável Target "SeriousDlqin2yrs".   
Problema de Classificação Binária.    
> Features (X): idade, renda, histórico de atrasos, etc. → o que eu SEI sobre a pessoa.  
> Target (y): SeriousDlqin2yrs → o que eu QUERO PREVER sobre a pessoa.  


## 6. Conceitos-chaves do problema

> **Probability of Default (PD)**  
É a probabilidade de um cliente não pagar sua dívida. Nesse caso, é a probabilidade de SeriousDlqin2yrs = 1.    

Cliente A: PD = 0.05 (5%)  → Baixo risco  
Cliente B: PD = 0.60 (60%) → Alto risco  

Bancos não querem apenas "Sim/Não", querem probabilidades, o que permite ordenar clientes por risco (scoring).

Possibilita decisões mais sofisticadas: 


PD < 10% → Aprovar com juros baixos  
PD 10-30% → Aprovar com juros altos  
PD > 30% → Nega  

> **Matriz de Confusão e Custos de Negócio**  
Uma tabela que mostra os **4 cenários possíveis** de predição em um problema de classificação binária.  

### Os 4 Cenários no Credit Scoring

| Caso | O que aconteceu | Custo/Impacto |
|------|----------------|---------------|
| **TN** (True Negative) | Cliente bom, aprovei ✅ | Lucro (juros pagos) 💰 |
| **TP** (True Positive) | Cliente ruim, neguei ✅ | Perdi negócio, mas evitei prejuízo |
| **FP** (False Positive) | Cliente **BOM**, mas neguei ❌ | Perdi cliente (custo de oportunidade) |
| **FN** (False Negative) | Cliente **RUIM**, mas aprovei ❌ | **PREJUÍZO GRANDE** 💸 (calote) |

### Por que importa para Credit Scoring?

**Os erros têm custos diferentes!**

- **FN (aprovar quem dá calote):** Você perde o valor emprestado + juros esperados
  - Exemplo de custo: -R$ 10.000
  
- **FP (negar quem pagaria):** Você perde apenas o lucro que teria
  - Exemplo de custo: -R$ 500

⚠️ **IMPORTANTE: FN pode ser ~20x mais caro que FP!**

### Implicações para o Modelo

1. **Minimizar FN é mais importante** que minimizar FP
2. Devemos ser **mais conservadores** na aprovação de crédito
3. É preferível **perder alguns bons clientes** a aprovar caloteiros
4. O **threshold de decisão** deve considerar esses custos assimétricos

> **Precision, Recall e F1-Score (em dados desbalanceados)**  
Em classes desbalanceadas, como esse dataset esperamos que ~93-94% não dêem calote e apenas ~6-7% dêem calote. 

> **PRECISION**  
Dos que eu PREVI como "vai dar calote", quantos REALMENTE deram?  
Precision = TP / (TP + FP)  

> **RECALL**   
Dos que REALMENTE deram calote, quantos eu CONSEGUI identificar?  
Recall = TP / (TP + FN)   

> **F1-Score**  
Média harmônica entre Precision e Recall    
F1 = 2 × (Precision × Recall) / (Precision + Recall)    



> **AUC-ROC (Area Under ROC Curve)**  
ROC = Receiver Operating Characteristic (curva)  
AUC = Area Under the Curve (área sob a curva ROC)  

**Interpretação do AUC:**

| AUC | Interpretação |
|-----|---------------|
| 1.0 | Perfeito (separa 100% corretamente) |
| 0.9-1.0 | Excelente |
| 0.8-0.9 | Bom |
| 0.7-0.8 | Razoável |
| 0.5 | Inútil (chute aleatório) |

**Por que importa?**
- Métrica padrão da indústria de crédito
- Independente do threshold escolhido
- Funciona bem com classes desbalanceadas
- Permite comparação justa entre modelos   

**Por que é melhor que Acurácia em dados desbalanceados?**

Em nosso dataset (~94% não dá calote, ~6% dá calote), um modelo 
"burro" que SEMPRE prediz "0" (não dá calote) teria:
- Acurácia: 94% (parece ótimo!)
- AUC: 0.5 (revela que é inútil)


**KS Statistic (Kolmogorov-Smirnov)**  
Métrica MUITO usada em credit scoring no mercado. Mede a máxima 
separação entre a distribuição de bons e maus pagadores.

**Interpretação:**

| KS (%) | Qualidade |
|--------|-----------|
| > 40% | Excelente |
| 30-40% | Bom |
| 20-30% | Razoável |
| < 20% | Fraco |

**Por que importa?**
- É a "linguagem do mercado" - bancos falam "nosso modelo tem KS de 35%"
- Complementa o AUC com perspectiva diferente
- Muito usado em relatórios de risco para reguladores
- Mede o "poder discriminatório" do modelo

**Relação com AUC:**
Ambos medem separação entre classes, mas de formas diferentes. 
Um modelo bom terá alto AUC E alto KS.




## 7. Entendendo Trade-offs

**Modelo Conservador (prioriza Recall):**    
✅ Alto Recall (pega quase todos os caloteiros)        
❌ Baixa Precision (rejeita muitos bons clientes também)  


**Modelo Liberal (prioriza Precision):**  
✅ Alta Precision (quando rejeita, quase sempre é caloteiro mesmo)  
❌ Baixo Recall (deixa muitos caloteiros passarem)    

  
**Objetivo:** Minimizar FN (não aprovar caloteiros)  
      ↓  
**Métrica:** Maximizar Recall  
      ↓  
**Estratégia:** Ser conservador (threshold baixo)  
      ↓  
**Consequência:** Vai rejeitar alguns bons clientes (↑ FP)  
      ↓  
**Trade-off aceitável:** Melhor perder negócio que tomar calote  

## 8. Resumo - Fase 1: Business Understanding

### O que definimos:

1. **Problema de Negócio:** Prever PD (Probability of Default) em 2 anos
2. **Tipo de Problema:** Classificação Binária (0 = não inadimple, 1 = inadimple)
3. **Variável Target:** `SeriousDlqin2yrs`
4. **Métrica Principal:** AUC-ROC (padrão da indústria)
5. **Métrica Crítica:** Recall (minimizar FN é prioridade)
6. **Métricas Complementares:** KS Statistic, Precision-Recall
7. **Trade-off Aceitável:** ↑ FP aceitável para ↓ FN
8. **Threshold:** Será ajustado na Fase 5 considerando custos assimétricos

### Critério de Sucesso para este Projeto:

- [ ] AUC-ROC > 0.80 (bom)
- [ ] KS > 30% (bom)
- [ ] Recall > 0.75 (pegar 75%+ dos caloteiros)
- [ ] Modelo explicável (para auditoria/regulação)
- [ ] Compreender todo o pipeline CRISP-DM
